In [ ]:
print("OK")

OK


In [ ]:
%pwd

'/Users/sachinchopra/Desktop/medical-chatbot/research'

In [ ]:
import os 
os.chdir("../")

In [ ]:
%pwd

'/Users/sachinchopra/Desktop/medical-chatbot'

In [ ]:
from langchain.document_loaders import PyPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

/opt/miniconda3/envs/medibot/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents 

In [ ]:
extracted_data = load_pdf_files("data")

In [ ]:
extracted_data

[Document(metadata={'producer': 'calibre 7.4.0', 'creator': 'calibre 7.4.0', 'creationdate': '2025-08-31T20:48:07+00:00', 'author': 'Tao Le', 'moddate': '2025-08-31T20:48:07+00:00', 'title': 'First Aid for the USMLE Step 2 CS', 'source': 'data/_OceanofPDF.com_First_Aid_for_the_USMLE_Step_2_CS_Fifth_Edition_-_tao_le.pdf', 'total_pages': 829, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'calibre 7.4.0', 'creator': 'calibre 7.4.0', 'creationdate': '2025-08-31T20:48:07+00:00', 'author': 'Tao Le', 'moddate': '2025-08-31T20:48:07+00:00', 'title': 'First Aid for the USMLE Step 2 CS', 'source': 'data/_OceanofPDF.com_First_Aid_for_the_USMLE_Step_2_CS_Fifth_Edition_-_tao_le.pdf', 'total_pages': 829, 'page': 1, 'page_label': '2'}, page_content='OceanofPDF .com'),
 Document(metadata={'producer': 'calibre 7.4.0', 'creator': 'calibre 7.4.0', 'creationdate': '2025-08-31T20:48:07+00:00', 'author': 'Tao Le', 'moddate': '2025-08-31T20:48:07+00:00', 'title': 'First Aid

In [ ]:
from typing import List 
from langchain.schema import Document



In [ ]:
def filter_to_minimal(docs:List[Document]) -> List[Document]:
    minimal_docs = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source":src}
                
            )
        )
    return minimal_docs

In [ ]:
minimal_data = filter_to_minimal(extracted_data)

In [ ]:
minimal_data

[Document(metadata={'source': 'data/_OceanofPDF.com_First_Aid_for_the_USMLE_Step_2_CS_Fifth_Edition_-_tao_le.pdf'}, page_content=''),
 Document(metadata={'source': 'data/_OceanofPDF.com_First_Aid_for_the_USMLE_Step_2_CS_Fifth_Edition_-_tao_le.pdf'}, page_content='OceanofPDF .com'),
 Document(metadata={'source': 'data/_OceanofPDF.com_First_Aid_for_the_USMLE_Step_2_CS_Fifth_Edition_-_tao_le.pdf'}, page_content='OceanofPDF .com'),
 Document(metadata={'source': 'data/_OceanofPDF.com_First_Aid_for_the_USMLE_Step_2_CS_Fifth_Edition_-_tao_le.pdf'}, page_content='Copyright © 2014 by Tao Le. All rights reserved. Except as permitted under\nthe United States Copyright Act of 1976, no part of this publication may be\nreproduced or distributed in any form or by any means, or stored in a\ndatabase or retrieval system, without the prior written permission of the\npublisher.\nISBN: 978-0-07-180933-7\nMHID:       0-07-180933-3\nThe material in this eBook also appears in the print version of this title:

In [ ]:
#dividing my data into chunks 
def text_splitter(minimal_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap=20
    )
    text_chunks = text_splitter.split_documents(minimal_data)
    return text_chunks


In [ ]:
text_chunks = text_splitter(minimal_data)

In [ ]:
len(text_chunks)

36497

In [ ]:
from langchain.embeddings import HuggingFaceBgeEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceBgeEmbeddings(
        model_name= model_name
    )
    return embeddings

In [ ]:
embedding = download_embeddings()

/var/folders/7l/w_rxr30j0t179mr0swktt1pc0000gn/T/ipykernel_2208/1575861042.py:5: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings(


In [ ]:
vector = embedding.embed_query("HI my name is Sachin")

In [ ]:
len(vector)

384

In [ ]:
from dotenv import load_dotenv
import os 
load_dotenv()

True

In [ ]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY 
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY 


In [ ]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
pc

In [ ]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws",region="us-east-1")
    )

index = pc.Index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embedding,
    index_name=index_name
)

In [ ]:
retreiver = docsearch.as_retriever(search_type = "similarity",search_kwargs={"k":3})

In [ ]:
retreived = retreiver.invoke("What is acne")
retreived

[Document(id='e1e87b12-0994-4142-b82e-368c9997c63e', metadata={'source': 'data/_OceanofPDF.com_CURRENT_Medical_Diagnosis_and_Treatment_2025_-_Maxine_A_Papadakis.pdf'}, page_content='mon identifiable cause. Acne may develop in patients who \nuse systemic corticosteroids or topical fluorinated corti -\ncosteroids on the face. Acne may be exacerbated or caused \nby cosmetic creams or oils as well as androgenic supple -\nments or masculinizing hormone therapy in transgender \nindividuals.\n » Clinical Findings\nThere may be mild tenderness, pain, or itching. The lesions \noccur mainly over the face, neck, upper chest, back, and'),
 Document(id='adfbe48c-3259-4025-9bb4-148394cdfbb8', metadata={'source': 'data/_OceanofPDF.com_CURRENT_Medical_Diagnosis_and_Treatment_2025_-_Maxine_A_Papadakis.pdf'}, page_content='mon identifiable cause. Acne may develop in patients who \nuse systemic corticosteroids or topical fluorinated corti -\ncosteroids on the face. Acne may be exacerbated or caused \nby 

In [ ]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o")

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "You are a medical assistant for question answering tasks"
    "Use the following pieces of retreived content to answer"
    "if you dont know say that you dont know"
    "answer in 1 para"
    "\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}")
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(chatModel,prompt)
rag_chain = create_retrieval_chain(retreiver,question_answer_chain)

In [55]:
response = rag_chain.invoke({"input": "do you know any anime betrayal"})
print(response["answer"])

I don't have specific examples of anime betrayals. If you're looking for information on popular anime moments involving betrayal or need a recommendation, you might want to explore forums or dedicated anime communities.
